# Übung 2: Extruder-Blockade mit KNN vorhersagen
**Internet of Production – KI, Woche 1** · ca. 25 min · Arbeit in Zweiergruppen

**Szenario:** Eine Extruder-Blockade legt Drucker 1 für Stunden still und beendet den laufenden Auftrag. In Übung 1 haben Sie gesehen, dass der Energieverbrauch je Minute vor einer Blockade ansteigt. Nun machen Sie aus dieser Beobachtung einen **Klassifikator**, der den Vorfertigungsleiter warnt.

**Aufgabe:** Entscheiden Sie für jede Druckminute: *Tritt innerhalb der nächsten 6 Stunden eine Extruder-Blockade auf?* (Klasse 1) oder nicht (Klasse 0). Nutzen Sie **K-Nearest-Neighbours (KNN)** auf drei aus dem Energieverbrauch abgeleiteten Merkmalen.

**Die Schritte 1 bis 3 sind vorbereitet** (Label, Merkmale, Aufteilung): Führen Sie sie aus und erklären Sie Ihrer Partnerin oder Ihrem Partner, was jeder Block tut (5 min). Die Lücken beginnen in Schritt 5. Danach arbeiten Sie an dem, worauf es ankommt: KNN, Skalierung und Metriken.

Jedes `___` ist eine Lücke.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay

DATA = Path('../../00_Data/printfarm')

# from exercise 1
def load_printer(k, data=DATA):
    d = pd.read_csv(data / f'Printer_{k}.csv', sep=';', decimal=',', low_memory=False).dropna(how='all')
    d['Timestamp'] = pd.to_datetime(d['Timestamp'], format='%d.%m.%Y %H:%M')
    d = d.set_index('Timestamp')
    d['Energy'] = d['Total_Power'].diff()
    return d

def load_errors(k, data=DATA):
    e = pd.read_csv(data / f'Printer_{k}_error.csv', sep=';').dropna(how='all')
    e['Timestamp'] = pd.to_datetime(e['Timestamp'], format='%d.%m.%Y %H:%M')
    return e

df = load_printer(1)
err = load_errors(1)
blockages = np.sort(err.loc[err['Error'] == 'Extruder blocked', 'Timestamp'].values)
print(len(df), 'minutes,', len(blockages), 'extruder blockages')

## Schritt 1: Das Label (vorbereitet)
Für jede Druckminute brauchen wir die **Zeit bis zur nächsten Blockade**. Die Klasse ist 1, wenn diese Zeit zwischen 0 und dem Vorhersagehorizont liegt.

In [ ]:
HORIZON = 360                                   # minutes: warn 6 hours ahead
p = df[df['State'] == 'printing'].copy()
p['E'] = p['Energy'] * 1000                     # Wh per minute (if Total_Power is in kWh)

idx = p.index.values
nxt = np.searchsorted(blockages, idx)           # index of the next blockage
minutes_to_blockage = np.where(nxt < len(blockages),
                               (blockages[np.minimum(nxt, len(blockages) - 1)] - idx) / np.timedelta64(1, 'm'), np.inf)
p['y'] = ((minutes_to_blockage > 0) & (minutes_to_blockage <= HORIZON)).astype(int)
print(p['y'].value_counts())

## Schritt 2: Merkmale aus dem Energieverbrauch (vorbereitet)
Eine einzelne Minute rauscht zu stark (Übung 1). Wir nutzen gleitende Mittelwerte über die letzten Druckminuten:
- `E30`: Mittelwert der letzten 30 Minuten (kurzfristig)
- `E120`: Mittelwert der letzten 120 Minuten
- `slope`: Änderung von `E120` über die letzten 240 Minuten (Trend)

Wir behalten jede 10. Minute, damit benachbarte Stichproben nicht nahezu identisch sind.

In [ ]:
p['E30'] = p['E'].rolling(30).mean()
p['E120'] = p['E'].rolling(120).mean()
p['slope'] = p['E120'] - p['E120'].shift(240)
features = ['E30', 'E120', 'slope']

data = p.dropna(subset=features).iloc[::10]
print(len(data), 'samples,', f"{data['y'].mean():.1%} class 1")
data[features + ['y']].describe().round(3)

## Schritt 3: Nach Zeit aufteilen
Trainieren Sie mit den ersten 70 %, testen Sie mit den letzten 30 %. Warum nicht zufällig?

In [ ]:
cut = int(len(data) * 0.7)
train, test = data.iloc[:cut], data.iloc[cut:]
X_train, y_train, X_test, y_test = train[features], train['y'], test[features], test['y']
print('train:', len(train), 'samples, class 1:', y_train.sum())
print('test :', len(test), 'samples, class 1:', y_test.sum())

## Schritt 4: Der Merkmalsraum
Stellen Sie die Testdaten in der Ebene `E120` und `slope` dar. Können Sie sich eine **Entscheidungsgrenze** zwischen den Klassen vorstellen? Wo überlappen sich die Klassen?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(test.loc[test['y'] == 0, 'E120'], test.loc[test['y'] == 0, 'slope'], s=6, color='#009EE0', alpha=.4, label='no blockage within 6 h')
ax.scatter(test.loc[test['y'] == 1, 'E120'], test.loc[test['y'] == 1, 'slope'], s=14, color='#EE7F00', label='blockage within 6 h')
ax.set_xlabel('E120 (Wh/min)'); ax.set_ylabel('slope (Wh/min)'); ax.legend(); plt.show()

## Schritt 5: Baseline und KNN
KNN nutzt Abstände, daher müssen die Merkmale **skaliert** werden. Probieren Sie mehrere Werte für `k` aus.

In [ ]:
def report(name, y_true, y_pred):
    print(f'{name:30s} accuracy {accuracy_score(y_true, y_pred):6.1%}   recall {recall_score(y_true, y_pred, zero_division=0):6.1%}   precision {precision_score(y_true, y_pred, zero_division=0):6.1%}')

base = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
report('baseline: never warn', y_test, base.predict(X_test))

for k in [5, 25, 75]:
    knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)).fit(X_train, y_train)
    report(f'KNN, k = {k}', y_test, knn.predict(X_test))

**Vergleichen Sie Accuracy mit Recall und Precision. Schlägt KNN die Baseline, und bei welcher Metrik? Wie verändert `k` das Ergebnis?**

Die Baseline warnt nie und erreicht etwa 96 % Accuracy bei 0 % Recall: Die Accuracy verdeckt das Problem, weil Klasse 1 selten ist (etwa 4 %). KNN steigert die Accuracy nur leicht (etwa 97 %), findet aber etwa 55–60 % der Blockaden (Recall) bei etwa 65–73 % echten Alarmen (Precision). Ein kleines `k` folgt einzelnen Punkten (verrauscht); ein großes `k` glättet, hier mit ähnlicher Güte für k = 25 bis 75.

## Schritt 6: Konfusionsmatrix und Warnungen je Blockade
Wie viele Blockaden des Testzeitraums werden durch mindestens eine Warnung in den 6 Stunden davor angekündigt? Wie viele Fehlalarme gibt es?

In [ ]:
best = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=25)).fit(X_train, y_train)
pred = best.predict(X_test)
ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=['no blockage', 'blockage < 6 h']).plot(cmap='Blues')
plt.show()

alarms = test.assign(alarm=pred)
test_blockages = [t for t in pd.to_datetime(blockages) if t >= test.index[0]]
announced = sum(alarms.loc[t - pd.Timedelta(hours=6): t, 'alarm'].any() for t in test_blockages)
print(f'blockages in the test period: {len(test_blockages)}, announced by a warning: {announced}')
print('false alarms (sampled minutes):', int(((alarms['alarm'] == 1) & (alarms['y'] == 0)).sum()))

## Ihre Bewertung (drei Sätze)
Ist die Warnung für den Vorfertigungsleiter nützlich? Was kostet ein Fehlalarm, was eine übersehene Blockade? Was begrenzt das Ergebnis?

Eine Warnung bis zu 6 Stunden vorher gibt Zeit, die Düse zwischen zwei Aufträgen zu warten; ein Fehlalarm kostet eine kurze Prüfung, eine übersehene Blockade einen gescheiterten Auftrag und Stunden Stillstand, daher ist eine geringe Precision akzeptabel, solange der Recall brauchbar ist. Begrenzt wird das Ergebnis durch die wenigen Blockaden (insgesamt 34, im Testzeitraum nur eine Handvoll), dadurch, dass nur Drucker 1 den Anstieg zeigt, und durch das Rauschen des Energiesignals. Nächster Schritt: Regression (Zeit bis zur Blockade) und weitere Merkmale (Temperatur) in Woche 3.

**Nach der Peer Instruction (Woche 2): Abstimmung überprüfen.** Vergleichen Sie die Accuracy **desselben Modells** auf den Trainingsdaten, mit denen es angepasst wurde, und auf den Testdaten, für `k = 1` und `k = 25`.

In [ ]:
for k in [1, 25]:
    knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)).fit(X_train, y_train)
    print(f'k = {k:2d}: accuracy on training data {accuracy_score(y_train, knn.predict(X_train)):7.2%}   on test data {accuracy_score(y_test, knn.predict(X_test)):7.2%}')
print(f'baseline (never warn) on test data: {accuracy_score(y_test, base.predict(X_test)):7.2%}')

**Warum liegt die Trainings-Accuracy bei `k = 1` fast bei 100 %, und warum sagt das nichts über die Güte des Modells?**

Jeder Trainingspunkt ist sein eigener nächster Nachbar (Abstand 0), `k = 1` gibt also einfach das gespeicherte Label zurück (hier 99,97 %; der Rest sind identische Punkte mit unterschiedlichem Label). Das Modell hat die Trainingsdaten auswendig gelernt. Auf neuen Daten erreicht es nur 95,9 %, kaum mehr als die Baseline von 95,8 %, während `k = 25` besser verallgemeinert (97,3 %). Die Accuracy muss daher immer auf Daten gemessen werden, die das Modell nicht gesehen hat.